# 🚀 Advanced FLAN-T5 Text-to-SQL Fine-Tuning (LoRA / rsLoRA / DoRA)

This notebook trains a specialized **FLAN-T5** model for Text-to-SQL using advanced fine-tuning techniques:
- **rsLoRA (Rank-Stabilized LoRA)** & **DoRA (Weight-Decomposed LoRA)**
- Target linear projections (`q`, `v`, `k`, `o`, `wi`, `wo`)
- Pad token label loss masking (`-100`)
- Mixed precision (`fp16`/`bf16`) and cosine annealing learning rate scheduler
- Exports trained LoRA adapters ready for the local FastAPI fallback!

In [ ]:
# 1. Install dependencies
!pip install -q transformers peft datasets evaluate accelerate bitsandbytes sentencepiece

In [ ]:
# 2. Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU. For faster training, switch Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 3. Clone or upload your dataset
# You can upload 'training/data/mixture_train.jsonl' directly to the Colab files panel on the left!
from pathlib import Path
data_path = Path("mixture_train.jsonl")
if not data_path.exists():
    print("Please upload mixture_train.jsonl to Colab files panel before proceeding.")
else:
    print(f"Found dataset: {data_path}")

In [ ]:
# 4. Launch Advanced Fine-Tuning
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

MODEL_NAME = "google/flan-t5-base"  # Or 'google/flan-t5-small'
OUTPUT_DIR = "./flan_t5_sql_lora"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Apply rsLoRA with targets on all linear layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "v", "k", "o", "wi_0", "wi_1", "wo"],
    use_rslora=True,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Dataset Preprocessing & Training
dataset = load_dataset("json", data_files="mixture_train.jsonl", split="train").shuffle(seed=42)
split_ds = dataset.train_test_split(test_size=0.05, seed=42)

def preprocess(batch):
    prompts = [f"Instruction: {i}\n{inp}\nSQL:" for i, inp in zip(batch["instruction"], batch["input"])]
    model_inputs = tokenizer(prompts, max_length=512, truncation=True)
    labels = tokenizer(text_target=batch["output"], max_length=256, truncation=True)
    cleaned_labels = [
        [tok if tok != tokenizer.pad_token_id else -100 for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = cleaned_labels
    return model_inputs

train_data = split_ds["train"].map(preprocess, batched=True, remove_columns=split_ds["train"].column_names)
eval_data = split_ds["test"].map(preprocess, batched=True, remove_columns=split_ds["test"].column_names)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100),
)

trainer.train()

In [ ]:
# 6. Save and Download Adapters
final_path = "./flan_t5_sql_lora/final"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

# Zip adapter folder for download
!zip -r flan_t5_sql_lora.zip ./flan_t5_sql_lora/final
print("Done! Download 'flan_t5_sql_lora.zip' and place in 'training/artifacts/' in your project.")